# Pseudo-Relevance Feedback (PRF)
---
Pseudo-Relevance Feedback, или сокращенно **PRF**, — это эвристический метод, используемый в информационном поиске для автоматического уточнения поискового запроса. Он основан на идее, что первые несколько документов, возвращенных при первоначальном поиске, скорее всего, релевантны, и их можно использовать для улучшения исходного запроса.

**Контекст**
При работе с системами информационного поиска пользователи часто сталкиваются с проблемой формулирования идеального запроса. Исходный запрос может быть слишком коротким, многозначным или не содержать всех необходимых ключевых слов, что приводит к неоптимальным результатам поиска.

**Идея метода**
Основная идея PRF заключается в *автоматической имитации* обратной связи с пользователем. Вместо того, чтобы просить пользователя вручную отмечать документы как релевантные или нерелевантные (как в традиционном Relevance Feedback), PRF *предполагает*, что документы, находящиеся в топе результатов начального поиска, действительно релевантны исходному запросу. Эти "псевдорелевантные" документы затем используются для расширения или перевзвешивания исходного запроса, что должно улучшить качество последующей выдачи.

**Постановка задачи**
Задача PRF состоит в повышении качества найденных документов для заданного пользовательского запроса $Q$ из коллекции документов $D = \{d_1, d_2, ..., d_N\}$. Цель — получить более точный или полный набор документов после выполнения начального поиска.

**Предшествующие методы**
*   **Initial Query Formulation:** Самый базовый подход, при котором пользователь вручную вводит запрос. Эффективность полностью зависит от формулировки запроса пользователем.
*   **Relevance Feedback (RF) / Rocchio Algorithm (1971):** Непосредственный предшественник PRF. В этом методе пользователи *явно* идентифицируют релевантные и нерелевантные документы из первоначального набора результатов поиска. Алгоритм Rocchio затем создает новый, расширенный вектор запроса, используя исходный запрос, релевантные документы и (опционально) нерелевантные документы. Ключевое архитектурное отличие RF от PRF заключается в необходимости *взаимодействия с человеком*, что делает его медленным и часто неприменимым в крупномасштабных системах.
*   **Query Expansion (QE):** Более широкая категория методов, часто включающая использование тезаурусов или предопределенных связей между терминами (например, WordNet) для добавления синонимов или связанных терминов к запросу. В отличие от PRF, эти методы обычно не зависят от *текущих* результатов поиска, а скорее от внешних баз знаний или статических статистик.

**Архитектура и компоненты**
PRF сам по себе не является нейронной архитектурой в современном понимании. Это многостадийный процесс, включающий:
*   **Систему первичного поиска:** Любая стандартная IR-система (например, использующая TF-IDF, BM25 (1994) или даже ранние модели Dense Retrieval) для выполнения первого прохода.
*   **Модуль представления запроса:** Преобразует текстовые запросы/документы в подходящее представление (например, вектор "мешка слов", эмбеддинги).
*   **Функцию ранжирования:** Вычисляет меру схожести между запросом и документами.
*   **Алгоритм уточнения запроса:** Основная логика PRF, обычно адаптация алгоритма Rocchio или методов статистического взвешивания терминов.

**Алгоритм работы**

1.  **Первичный поиск:**
    *   Получив исходный пользовательский запрос $Q_{orig}$, выполняется стандартный поиск по коллекции документов $D$.
    *   Возвращается упорядоченный список из топ-K документов, $D_{topK} = \{d_1, d_2, ..., d_K\}$, на основе их оценок релевантности.

2.  **Гипотеза псевдорелевантности:**
    *   Предполагается, что первые $N$ документов из $D_{topK}$ (где $N \le K$, обычно $N$ мало, например, 5-10) являются "псевдорелевантными" информационной потребности пользователя. Этот набор называется $D_{pseudo\_rel}$.

3.  **Расширение/перевзвешивание запроса:**
    *   Извлекаются важные термины или признаки из документов в $D_{pseudo\_rel}$.
    *   Эти термины/признаки объединяются с исходным запросом $Q_{orig}$ для создания нового, расширенного запроса $Q_{new}$.
    *   Наиболее распространенный метод для этого шага — вариация алгоритма Rocchio:
        *   Если используются векторные представления терминов: $Q_{new} = \alpha Q_{orig} + \beta \sum_{d \in D_{pseudo\_rel}} d$. ($\alpha$ и $\beta$ — параметры взвешивания. Терм $\gamma$ для нерелевантных документов обычно опускается в PRF, так как явных нерелевантных документов нет).
        *   Для моделей, использующих Dense Embeddings (например, после появления DPR (2020)), это может включать усреднение эмбеддингов $Q_{orig}$ и псевдорелевантных документов: $E_{Q_{new}} = \text{avg}(E_{Q_{orig}}, E_{d_1}, E_{d_2}, ..., E_{d_N})$.
    *   Другие методы включают статистические подходы, такие как добавление терминов с высокими оценками TF-IDF из $D_{pseudo\_rel}$, которых нет в $Q_{orig}$, или использование моделей отклонения от случайности для выявления значимых терминов.

4.  **Второй проход поиска:**
    *   Расширенный запрос $Q_{new}$ используется для выполнения второго поиска по *всей* коллекции документов $D$.
    *   Возвращаются новые, переранжированные топ-документы в качестве окончательного результата поиска.

**Инференс**
PRF обычно применяется динамически во время обработки запроса. Пользователь отправляет запрос, система выполняет первоначальный поиск, затем PRF автоматически уточняет запрос, после чего выполняется второй поиск, и пользователю представляются улучшенные результаты. Этот процесс полностью автоматический и прозрачный для пользователя.

**Результаты**
*   **Улучшение полноты и точности:** PRF последовательно демонстрирует улучшения как в полноте (обнаружение большего количества релевантных документов), так и в точности (меньше нерелевантных документов в топе результатов) по сравнению с использованием только исходного запроса, особенно для коротких или неоднозначных запросов.
*   **Устойчивость:** Метод помогает смягчить проблему "несоответствия словарного запаса" (vocabulary mismatch), когда термины, используемые в запросе, не совпадают идеально с терминами в релевантных документах. Расширяя запрос терминами из псевдорелевантных документов, PRF сокращает этот разрыв.
*   **Компромисс производительности:** Преимущество достигается за счет увеличения вычислительных затрат из-за двухэтапного процесса поиска. Выбор $K$ (количество документов для первичного поиска) и $N$ (количество псевдорелевантных документов) существенно влияет на производительность и эффективность. Слишком малое $N$ может не захватить достаточно полезного сигнала; слишком большое $N$ может внести шум от действительно нерелевантных документов.
*   **Риск Query Drift:** Если изначально высокоранжированные документы на самом деле нерелевантны, PRF может столкнуться с проблемой **query drift** (дрейф запроса), когда уточненный запрос отдаляется от реальной информационной потребности пользователя, что приводит к худшим результатам, чем при использовании исходного запроса. Это является основной проблемой PRF.
*   **PRF в нейронных сетях:** С появлением Dense Retrieval (например, DPR (2020)), принципы PRF были адаптированы. Усреднение плотных эмбеддингов топ-K найденных документов с эмбеддингом исходного запроса часто демонстрирует прирост, подтверждая полезность подхода даже в современных нейронных IR-системах. Например, некоторые исследования показывают прирост на 2-5% в метриках MRR (Mean Reciprocal Rank) или Recall@K на стандартных IR-бенчмарках, таких как MS MARCO, при использовании нейронного PRF по сравнению с базовым нейронным поиском.

## 📝 Критический анализ

```markdown
# Pseudo-Relevance Feedback (PRF)
---
Pseudo-Relevance Feedback (PRF) — это метод в информационном поиске для автоматического уточнения запроса. Он предполагает, что первые несколько документов, возвращенных при первоначальном поиске, релевантны и могут улучшить исходный запрос.

**Контекст**
Пользователи часто сталкиваются с проблемой формулирования идеального запроса, что приводит к неоптимальным результатам поиска.

**Идея метода**
PRF автоматически имитирует обратную связь, предполагая, что топовые документы начального поиска релевантны. Эти "псевдорелевантные" документы используются для расширения или перевзвешивания исходного запроса.

**Постановка задачи**
Цель PRF — улучшить качество найденных документов для заданного запроса $Q$ из коллекции документов $D$.

**Предшествующие методы**
- **Initial Query Formulation:** Пользователь вручную вводит запрос.
- **Relevance Feedback (RF) / Rocchio Algorithm (1971):** Пользователи явно идентифицируют релевантные документы, создавая новый запрос.
- **Query Expansion (QE):** Использует внешние базы знаний для добавления терминов к запросу.

**Архитектура и компоненты**
PRF — это многостадийный процесс:
- **Система первичного поиска:** Использует стандартные IR-системы.
- **Модуль представления запроса:** Преобразует запросы в векторные представления.
- **Функция ранжирования:** Оценивает схожесть между запросом и документами.
- **Алгоритм уточнения запроса:** Адаптация алгоритма Rocchio или статистическое взвешивание терминов.

**Алгоритм работы**

1. **Первичный поиск:**
   - Выполняется стандартный поиск по запросу $Q_{orig}$.
   - Возвращается топ-K документов $D_{topK}$.

2. **Гипотеза псевдорелевантности:**
   - Первые $N$ документов из $D_{topK}$ считаются "псевдорелевантными".

3. **Расширение/перевзвешивание запроса:**
   - Извлекаются важные термины из $D_{pseudo\_rel}$.
   - Создается новый запрос $Q_{new}$, например, с помощью алгоритма Rocchio.

4. **Второй проход поиска:**
   - Используется $Q_{new}$ для повторного поиска по коллекции $D$.

**Инференс**
PRF применяется динамически: запрос уточняется автоматически, и выполняется второй поиск для улучшения результатов.

<img src="img/img.png" width=500>

**Результаты**
- **Улучшение полноты и точности:** PRF улучшает полноту и точность по сравнению с исходным запросом.
- **Устойчивость:** Снижает проблему "несоответствия словарного запаса".
- **Компромисс производительности:** Двухэтапный процесс увеличивает вычислительные затраты.
- **Риск Query Drift:** Нерелевантные документы могут ухудшить результаты.
- **PRF в нейронных сетях:** Усреднение эмбеддингов топ-K документов улучшает метрики, такие как MRR и Recall@K, на 2-5% на бенчмарках, таких как MS MARCO.
```

## 💻 Пример кода

Иллюстративный Python пример, демонстрирующий основные концепции:

In [ ]:
# Импортируем необходимые библиотеки
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# Пример коллекции документов
documents = [
    "The quick brown fox jumps over the lazy dog",
    "Never jump over the lazy dog quickly",
    "A fast brown fox leaps over a sleepy dog",
    "The quick brown fox is fast and agile",
    "Dogs are loyal and friendly animals"
]

# Исходный пользовательский запрос
query = "quick fox"

# Шаг 1: Первичный поиск
# Используем TF-IDF для векторизации документов и запроса
vectorizer = TfidfVectorizer()
doc_vectors = vectorizer.fit_transform(documents)
query_vector = vectorizer.transform([query])

# Вычисляем косинусное сходство между запросом и документами
similarities = cosine_similarity(query_vector, doc_vectors).flatten()

# Получаем топ-K документов (например, K=3)
K = 3
top_k_indices = similarities.argsort()[-K:][::-1]
top_k_documents = [documents[i] for i in top_k_indices]

print("Top-K Documents from Initial Search:")
for doc in top_k_documents:
    print(doc)

# Шаг 2: Гипотеза псевдорелевантности
# Предполагаем, что все топ-K документы релевантны
pseudo_relevant_docs = top_k_documents

# Шаг 3: Расширение/перевзвешивание запроса
# Извлекаем термины из псевдорелевантных документов
pseudo_relevant_vectors = doc_vectors[top_k_indices]

# Используем алгоритм Rocchio для создания нового запроса
alpha = 1.0  # вес исходного запроса
beta = 0.75  # вес псевдорелевантных документов

# Усредняем векторы псевдорелевантных документов
pseudo_relevant_mean = pseudo_relevant_vectors.mean(axis=0)

# Создаем новый вектор запроса
new_query_vector = alpha * query_vector + beta * pseudo_relevant_mean

# Шаг 4: Второй проход поиска
# Вычисляем косинусное сходство между новым запросом и документами
new_similarities = cosine_similarity(new_query_vector, doc_vectors).flatten()

# Получаем новые топ-документы
new_top_k_indices = new_similarities.argsort()[-K:][::-1]
new_top_k_documents = [documents[i] for i in new_top_k_indices]

print("\nTop-K Documents after PRF:")
for doc in new_top_k_documents:
    print(doc)
```

### Объяснение ключевых моментов:

1. **Первичный поиск**: Мы используем TF-IDF для векторизации документов и запроса, а затем вычисляем косинусное сходство, чтобы получить топ-K документов.

2. **Гипотеза псевдорелевантности**: Предполагаем, что все топ-K документов релевантны, и используем их для уточнения запроса.

3. **Расширение/перевзвешивание запроса**: Применяем алгоритм Rocchio, чтобы создать новый вектор запроса, который учитывает как исходный запрос, так и псевдорелевантные документы.

4. **Второй проход поиска**: Используем новый вектор запроса для повторного поиска, чтобы получить улучшенные результаты.

Этот пример иллюстрирует, как PRF может автоматически улучшить результаты поиска без явного взаимодействия с пользователем, используя предположение о релевантности топ-K документов.